In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from brax import envs
from typing import Any, Tuple, List
from custom_types import Params, RNGKey, Env, EnvState
from flax import serialization
from networks import GCMLP, GC_PPO_Policy, ComplexGCPPO_Policy
from trajectory_encoder import TaskState, CircularEncoder, LineEncoder
from buffer import PPOTransition

import matplotlib.pyplot as plt
%matplotlib inline

from skill_tools import evaluate_deviation, build_line_task_state, build_circle_task_state, calculate_return

In [ ]:
seed = 42
# seed = 4444
loop_random_key = jax.random.PRNGKey(seed)
loop_random_key, subkey = jax.random.split(loop_random_key)

env = envs.create(env_name="ant", episode_length=4096, backend="mjx", auto_reset=True)

# task_encoder = LineEncoder(v_max=3.0)
# build_task_state = build_line_task_state 


task_encoder = CircularEncoder(v_max=3.0)
# build_task_state = build_circle_task_state

policy_network = GC_PPO_Policy(
    hidden_layer_sizes=(64, 64),
    action_dim=env.action_size,
    initial_std=0.1 * jnp.ones(env.action_size),
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2)),
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    activation=nn.softplus,
    # activation=nn.relu,
    final_activation=jnp.tanh,
    learnable_std=False,
    # learnable_std=True,
)


In [ ]:
fake_obs = jnp.zeros(shape=(env.observation_size,))
fake_zs = jnp.zeros(shape=(task_encoder.z_dim,))
dummy_policy_params = policy_network.init(subkey, obs=fake_obs, z=fake_zs)

Load Model

In [ ]:
with open("./circle/circle_model.msgpack", "rb") as f:
    model_bytes_data = f.read()

# with open("./line/line_model.msgpack", "rb") as f:
#     model_bytes_data = f.read()

# with open("./line_model_upgrade.msgpack", "rb") as f:
#     model_bytes_data = f.read()

# with open("line_model_complex.msgpack", "rb") as f:
#     model_bytes_data = f.read()
    
model_params = serialization.from_bytes(dummy_policy_params, model_bytes_data)

In [ ]:

@jax.jit
def rollout_fn(
    policy_params: Params,
    starting_states: EnvState,
    task_states: TaskState,
    keys: RNGKey,
    ) -> PPOTransition:

    def play_step_fn(
        carry: Tuple[EnvState, TaskState, RNGKey],
        ) -> Tuple[Tuple, PPOTransition]:
        
        state, task_state, key = carry
        key, subkey = jax.random.split(key)

        action_mean, _ = policy_network.apply(policy_params, state.obs, task_state.z)
        candidate_action_noise = 0.025 * jax.random.normal(subkey, action_mean.shape)
        action = jnp.clip(action_mean + candidate_action_noise, -1.0, 1.0)
    
        # step env  ->  next_state, fitness, 
        next_state = env.step(state, action)
        truncation = next_state.info['truncation']
        done = next_state.done - truncation
        fitness_rewards = jnp.array([next_state.reward - next_state.metrics["x_velocity"]])

        # step task 
        next_task_state = task_encoder.step(
            task_state, 
            next_state.pipeline_state.x.pos[0, :2] - state.pipeline_state.x.pos[0, :2], # displacement
            )
        
        reward, fail = evaluate_deviation(next_task_state.deviation)
        
        # record
        transition = PPOTransition(
            obs=state.obs,
            actions=action,
            tasks=task_state.z, 
            log_likelihood=jnp.zeros((1,)),
            rewards=jnp.array([reward]),
            fitness_rewards=fitness_rewards,
            td_lambda_returns=jnp.zeros((1,)),
            gaes=jnp.zeros(shape=(1,)),
            dones=jnp.where(done + fail > 0.9, jnp.ones(shape=(1,)), jnp.zeros(shape=(1,))),
            # dones=jnp.where(done > 0.9, jnp.ones(shape=(1,)), jnp.zeros(shape=(1,))),
            truncations=jnp.array([truncation]),
            weights=jnp.zeros(shape=(1,)),
            )

        return (next_state, next_task_state, key), transition

    final_carry, transitions = jax.lax.scan(
        lambda x, _: jax.vmap(play_step_fn)(x),
        (starting_states, task_states, keys),
        length=512,
    )
    # final_states, final_task_state, _ = final_carry

    return transitions

Plot heatmap of angle versus speed

In [ ]:
angles = jnp.reshape(jnp.arange(64) * jnp.pi/32, (-1, 1)) # in radius
speeds = jnp.linspace(1.0, 3.0, 32)
repeat_num = 32
rs = jnp.reshape(jnp.arange(64) / 4 + 1, (-1, 1))

In [ ]:
seed = 4242
loop_random_key = jax.random.PRNGKey(seed)
loop_random_key, subkey = jax.random.split(loop_random_key)
subkeys = jax.random.split(subkey, num=2048)

fixed_state = env.reset(subkey)
initial_states = jax.vmap(lambda x: fixed_state)(subkeys)

# initial_states = jax.vmap(env.reset)(subkeys)

In [ ]:
@jax.jit
def evaluate_tasks(carry: Tuple, speed):
    key = carry
    key, subkey = jax.random.split(key)
    subkeys = jax.random.split(subkey, num=2048)

    initial_vs = initial_states.obs[:, 13: 15]
    # theta = jnp.reshape(jnp.tile(angles, (1, repeat_num)), (-1,)) # (batch,)
    # task_states = jax.vmap(build_line_task_state, in_axes=(0, 0, 0, None))(
    #     initial_vs, 
    #     theta, 
    #     subkeys, 
    #     speed)
    
    r_values = jnp.reshape(jnp.tile(rs, (1, repeat_num)), (-1,)) # (batch,)
    task_states = jax.vmap(build_circle_task_state, in_axes=(0, 0, 0, None))(
        initial_vs, 
        r_values, 
        subkeys, 
        speed)
    
    key, subkey = jax.random.split(key)
    subkeys = jax.random.split(subkey, num=2048)
    
    transitions = rollout_fn(
        model_params,
        initial_states,
        task_states,
        subkeys
    )

    bd_values = calculate_return(transitions.rewards, transitions.dones) # batch x 1
    bd_values = jnp.mean(jnp.reshape(bd_values, (-1, repeat_num)), axis=1) # (32,)
    
    fitness_values = calculate_return(transitions.fitness_rewards, transitions.dones) # batch x 1
    fitness_values = jnp.mean(jnp.reshape(fitness_values, (-1, repeat_num)), axis=1) # (32,)

    lifespans = calculate_return(jnp.ones_like(transitions.rewards), transitions.dones) # batch x 1
    lifespans = jnp.mean(jnp.reshape(lifespans, (-1, repeat_num)), axis=1) # (32,)

    return key, (bd_values, fitness_values, lifespans)

In [ ]:

_, (bd_values, fitness_values, lifespans) = jax.lax.scan(
    evaluate_tasks,
    loop_random_key,
    speeds,
)


In [ ]:
print(bd_values.shape, fitness_values.shape)

In [ ]:
plt.imshow(bd_values[::-1, :])
plt.colorbar()
# plt.savefig(f"./capability/line/{seed}_eval.png")

speed_ticks = [31, 23, 15, 7, 0]
speed_labels = ['1.0', '1.5', '2.0', '2.5', '3.0']

radius_ticks = [0, 15, 31, 47, 63]
radius_labels = ["1.0", "5.0", "9.0", "13.0", "17.0"]

plt.xticks(radius_ticks, radius_labels)
plt.yticks(speed_ticks, speed_labels)
plt.xlabel("radius", fontsize=12)
plt.ylabel("speed", fontsize=12)
plt.savefig(f"./capability/circle/{seed}_eval.png")
# plt.show()

In [ ]:
ability = jnp.clip((bd_values[::-1, :] - 0.9 * 512) / (0.05*512), 0.0, 1.0)


plt.imshow(ability)
plt.colorbar()
speed_ticks = [31, 23, 15, 7, 0]
speed_labels = ['1.0', '1.5', '2.0', '2.5', '3.0']

radius_ticks = [0, 15, 31, 47, 63]
radius_labels = ["1.0", "5.0", "9.0", "13.0", "17.0"]

plt.xticks(radius_ticks, radius_labels)
plt.yticks(speed_ticks, speed_labels)
plt.xlabel("radius", fontsize=12)
plt.ylabel("speed", fontsize=12)
# plt.savefig(f"./capability/line/{seed}_ability.png")

plt.savefig(f"./capability/circle/{seed}_ability.png")
# plt.show()

In [ ]:
# from brax.io import html
# from IPython.display import HTML, display

In [ ]:
# def play_step_fn(state, task_state):

#     action, _ = policy_network.apply(model_params, state.obs, task_state.z)

#     next_state = env.step(state, action)
#     truncation = next_state.info['truncation']
#     done = next_state.done - truncation

#     next_task_state = task_encoder.step(
#         task_state, 
#         jnp.where(
#             done, 
#             -task_state.deviation, 
#             next_state.pipeline_state.x.pos[0, :2] - state.pipeline_state.x.pos[0, :2],
#         ), # displacement
#         )

#     return next_state, next_task_state, done

In [ ]:
# def run_rollout(initial_state, theta):
#     states = []
#     infos = []
#     # key, subkey = jax.random.split(key)
    
#     state = initial_state

#     initial_vs = initial_state.obs[13: 15]
#     initial_vs = jnp.where(initial_vs > 0, initial_vs + 1e-3, initial_vs - 1e-3) # (batch x 2)
#     unit_v = initial_vs / jnp.sqrt(jnp.sum(initial_vs**2, axis=-1, keepdims=True)) # (batch x 2)

#     task_state = build_task_state(unit_v, initial_state.obs[13: 15], theta, subkeys, 3.0)

    
#     for i in range(500):
#         states.append(state)

#         state, task_state, done = jax.jit(play_step_fn)(state, task_state)
#         infos.append(done) # record done info

#         # print(int(i + 1), "/ 500")
        
#     return states, infos

In [ ]:
# # 采样任务并记录轨迹
# print("simulating trajectory...")
# trajectory_states, trajectory_infos = run_rollout(initial_state, theta)

In [ ]:
# print(initial_states.obs.shape)

In [ ]:
# print("visualizing") 
# vis_html = html.render(env.sys, [s.pipeline_state for s in trajectory_states])

In [ ]:
# with open("visualize.html", "w") as f:
#     f.write(vis_html)